# Separable Box-Constrained Sum-Equality QP as a Neural Network Layer

This notebook demonstrates a **closed-form (via 1D root-finding) QP layer** for problems of the form

\[
\min_{x\in\mathbb{R}^N}\;\sum_{i=1}^N \left(\gamma_i x_i^2 + \beta_i x_i\right)
\quad\text{s.t.}\quad
\sum_{i=1}^N x_i=\xi,\;\; m_i \le x_i \le M_i.
\]

Ignoring constants, the objective can be rewritten as a **weighted projection**:

\[
\min_x \sum_i \gamma_i (x_i - z_i)^2 \quad\text{where}\quad z_i=-\frac{\beta_i}{2\gamma_i},\;\gamma_i>0.
\]

The layer computes

\[
x^\*(z) = \operatorname{clip}\!\left(z + \frac{\lambda}{2\gamma},\;[m,M]\right),
\]
where the scalar \(\lambda\) is chosen so that \(\sum_i x_i=\xi\).  
Because \(\sum_i \operatorname{clip}(\cdot)\) is monotone in \(\lambda\), we can solve \(\lambda\) robustly via **bisection**.

---

## What to look for in the demos

For each use case, we show:

- **Feasibility**: max violation of bounds and sum constraint (should be small).
- **Reference agreement**: compare a *fast* solver (e.g. 40 bisection steps) to a *high-accuracy* solver (e.g. 200 steps).
- **Distribution + sparsity**: histograms (often with log scales), support-size distributions, sorted-profile quantile bands.
- **Interpretability**: plots that show *piecewise* behavior as active sets change.

> Note: Many outputs are **exactly zero** (sparsity), which is why log plots need careful handling (zeros cannot be log-transformed directly).


In [ ]:

import os
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from separable_qp_layer import (
    SeparableQPProjection,
    SeparableQPSolveFromBeta,
    FixedZProjection,
)

def set_seed(seed: int = 0) -> None:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(0)

@dataclass(frozen=True)
class Cfg:
    device: str = "cpu"
    dtype: torch.dtype = torch.float32
    B: int = 256
    eps_log: float = 1e-12
    out_dir: str = "demo_outputs_notebook"

cfg = Cfg()
Path(cfg.out_dir).mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "figure.facecolor": "white",
})

def to_np(x: torch.Tensor) -> np.ndarray:
    return x.detach().cpu().numpy()

def savefig(name: str) -> None:
    path = Path(cfg.out_dir) / name
    plt.tight_layout()
    plt.savefig(path, dpi=170)
    print(f"saved: {path}")

def row_stats(x: torch.Tensor, eps: float = 1e-12) -> Dict[str, float]:
    x = x.detach()
    row_sum = x.sum(dim=-1)
    l0 = (x > 0).sum(dim=-1).float()
    top1 = x.max(dim=-1).values
    p = torch.clamp(x, min=eps)
    ent = -(p * torch.log(p)).sum(dim=-1)
    return {
        "mean_value": float(x.mean().item()),
        "std_value": float(x.std().item()),
        "min_value": float(x.min().item()),
        "max_value": float(x.max().item()),
        "mean_row_sum": float(row_sum.mean().item()),
        "std_row_sum": float(row_sum.std().item()),
        "mean_L0": float(l0.mean().item()),
        "std_L0": float(l0.std().item()),
        "mean_top1": float(top1.mean().item()),
        "mean_entropy": float(ent.mean().item()),
    }

def feasibility_stats(x: torch.Tensor, m: float, M: float, xi: float) -> Dict[str, float]:
    x = x.detach()
    row_sum = x.sum(dim=-1)
    sum_err = row_sum - xi
    below = (m - x).clamp(min=0).max().item()
    above = (x - M).clamp(min=0).max().item()
    return {
        "mean_abs_sum_err": float(sum_err.abs().mean().item()),
        "max_abs_sum_err": float(sum_err.abs().max().item()),
        "max_below_m": float(below),
        "max_above_M": float(above),
    }

def simplex_projection_reference(v: torch.Tensor, s: float = 1.0) -> torch.Tensor:
    if v.dim() == 1:
        v = v.unsqueeze(0)
    B, N = v.shape
    u, _ = torch.sort(v, dim=-1, descending=True)
    cssv = torch.cumsum(u, dim=-1) - s
    ind = torch.arange(1, N + 1, device=v.device, dtype=v.dtype).view(1, -1)
    cond = u - cssv / ind > 0
    rho = cond.to(torch.int64).sum(dim=-1) - 1
    rho = torch.clamp(rho, min=0)
    theta = cssv[torch.arange(B, device=v.device), rho] / (rho.to(v.dtype) + 1.0)
    return torch.clamp(v - theta.unsqueeze(-1), min=0.0)

def plot_hist(series: Dict[str, torch.Tensor], title: str, *, bins: int = 100, density: bool = True,
              logy: bool = False, xlim: Tuple[float, float] | None = None) -> None:
    plt.figure(figsize=(8, 4.6))
    for name, x in series.items():
        plt.hist(to_np(x.reshape(-1)), bins=bins, density=density, alpha=0.55, label=name, histtype="stepfilled")
    if logy:
        plt.yscale("log")
    if xlim is not None:
        plt.xlim(*xlim)
    plt.title(title)
    plt.legend()

def plot_sorted_quantile_band(x: torch.Tensor, title: str, *, q_lo=0.05, q_hi=0.95, q_med=0.5, k_max: int | None = None,
                              logy: bool = True) -> None:
    xs = torch.sort(x.detach(), dim=-1, descending=True).values
    if k_max is not None:
        xs = xs[:, :k_max]
    ql = torch.quantile(xs, q_lo, dim=0)
    qm = torch.quantile(xs, q_med, dim=0)
    qh = torch.quantile(xs, q_hi, dim=0)

    plt.figure(figsize=(8, 4.6))
    idx = np.arange(xs.shape[-1])
    plt.fill_between(idx, to_np(ql), to_np(qh), alpha=0.25, label=f"q{q_lo:.2f}-q{q_hi:.2f}")
    plt.plot(idx, to_np(qm), linewidth=2.0, label=f"median (q{q_med:.2f})")
    if logy:
        plt.yscale("log")
    plt.title(title)
    plt.xlabel("sorted index")
    plt.ylabel("value")
    plt.legend()

def plot_support_size(x: torch.Tensor, title: str, tol: float = 1e-12) -> None:
    l0 = (x.detach() > tol).sum(dim=-1).cpu().numpy()
    plt.figure(figsize=(7.8, 4.4))
    plt.hist(l0, bins=40, alpha=0.7)
    plt.title(title)
    plt.xlabel("support size (# nonzeros)")
    plt.ylabel("count")

def plot_log10_positive_with_zero_mass(x: torch.Tensor, title: str, *, eps: float = 1e-12, bins: int = 90) -> None:
    x = x.detach().reshape(-1)
    mask_pos = x > eps
    frac_zero = 1.0 - float(mask_pos.float().mean().item())
    x_pos = x[mask_pos]
    y = torch.log10(x_pos)

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), gridspec_kw={"width_ratios": [1, 3]})
    axes[0].bar(["<=eps"], [frac_zero], alpha=0.7)
    axes[0].set_ylim(0, 1)
    axes[0].set_title(f"zero mass\\nfrac={frac_zero:.3f}")
    axes[0].set_ylabel("fraction")

    axes[1].hist(to_np(y), bins=bins, density=True, alpha=0.65)
    axes[1].set_title("log10(positive values) density")
    axes[1].set_xlabel("log10(value)")
    fig.suptitle(title)
    fig.tight_layout()

def plot_error_distributions(err: torch.Tensor, title: str, *, eps: float = 1e-18) -> None:
    e = err.detach().reshape(-1).cpu()
    q = torch.quantile(e, 0.999).item()
    q = max(q, 1e-16)

    fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.2))
    axes[0].hist(e.numpy(), bins=100, density=True, alpha=0.65)
    axes[0].set_yscale("log")
    axes[0].set_xlim(0, q * 1.05)
    axes[0].set_title("error density (log-y), zoomed")
    axes[0].set_xlabel("|error|")

    y = torch.log10(e + eps)
    axes[1].hist(y.numpy(), bins=110, density=True, alpha=0.65)
    axes[1].set_title(f"log10(error+{eps:g}) density")
    axes[1].set_xlabel("log10(|error| + eps)")

    fig.suptitle(title)
    fig.tight_layout()


## Use case 1 — Simplex-like activation vs Softmax

**Setup:** \(m=0\), \(M=1\), \(\xi=1\), \(\gamma=1\).  
This produces a vector in \([0,1]^N\) that sums exactly to 1. It acts like a “probability vector” **but is often sparse** (many exact zeros).

**References used here**
- **QP-high-iter:** same algorithm but with many more bisection steps (numerical gold standard).
- **Simplex-sort (Duchi):** known sorting-based projection onto the simplex.

**What to expect**
- QP projection is typically *much* sparser than softmax.
- Softmax never produces exact zeros.


In [ ]:

B, N = cfg.B, 100
z = torch.randn(B, N, dtype=cfg.dtype)

proj_fast = SeparableQPProjection(n=N, bisection_iters=40)
proj_ref = SeparableQPProjection(n=N, bisection_iters=220)

x_qp = proj_fast(z)
x_qp_ref = proj_ref(z)
x_sm = F.softmax(z, dim=-1)
x_sort_ref = simplex_projection_reference(z, s=1.0)

print("QP stats:", row_stats(x_qp, eps=cfg.eps_log))
print("Softmax stats:", row_stats(x_sm, eps=cfg.eps_log))
print("QP feasibility:", feasibility_stats(x_qp, 0.0, 1.0, 1.0))
print("Fast vs high-iter mean/max abs:", float((x_qp-x_qp_ref).abs().mean()), float((x_qp-x_qp_ref).abs().max()))
print("QP vs simplex-sort mean/max abs:", float((x_qp-x_sort_ref).abs().mean()), float((x_qp-x_sort_ref).abs().max()))


### Value distributions

We show:
- a **log-y** histogram (still includes zeros),
- and a **log10(positive)** histogram that *separates zero mass explicitly*.


In [ ]:

plot_hist({"QP": x_qp, "Softmax": x_sm},
          "Simplex-like: value distribution (linear)", bins=120, logy=False, xlim=(0.0, 0.25))
savefig("simplex_hist_linear.png")
plt.show()

plot_hist({"QP": x_qp, "Softmax": x_sm},
          "Simplex-like: value distribution (log-y)", bins=160, logy=True, xlim=(0.0, 0.25))
savefig("simplex_hist_logy.png")
plt.show()

plot_log10_positive_with_zero_mass(x_qp, "QP simplex projection: log10(value) with zero-mass separation", eps=cfg.eps_log)
savefig("simplex_qp_log10_positive.png")
plt.show()


### Sorted profiles (improved quantile band)

Improvement over the earlier plot:
- we show only the **top-40** sorted coordinates (where the mass lives),
- and we use a stable quantile band \([q_{0.05}, q_{0.95}]\) plus the median.


In [ ]:

plot_sorted_quantile_band(x_qp, "QP simplex projection: sorted profile quantile band (top-40)", k_max=40, logy=True)
savefig("simplex_sorted_profiles_qp_quantiles_improved.png")
plt.show()

plot_sorted_quantile_band(x_sm, "Softmax: sorted profile quantile band (top-40)", k_max=40, logy=True)
savefig("simplex_sorted_profiles_softmax_quantiles_top40.png")
plt.show()

plot_support_size(x_qp, "QP simplex projection: support size distribution")
savefig("simplex_support_size_qp.png")
plt.show()


## Use case 2 — k-hot / budgeted gating

**Setup:** \(m=0\), \(M=1\), \(\xi=k\).  
Continuous relaxation of selecting ~\(k\) items.


In [ ]:

B, N = cfg.B, 200
k = 10.0
z = torch.randn(B, N, dtype=cfg.dtype)

proj_fast = SeparableQPProjection(n=N, xi=k, bounds_mode="fixed", m_init=0.0, M_init=1.0, bisection_iters=40)
proj_ref  = SeparableQPProjection(n=N, xi=k, bounds_mode="fixed", m_init=0.0, M_init=1.0, bisection_iters=240)

x = proj_fast(z)
x_ref = proj_ref(z)

print("k-hot stats:", row_stats(x, eps=cfg.eps_log))
print("Feasibility:", feasibility_stats(x, 0.0, 1.0, k))
print("Fast vs high-iter mean/max abs:", float((x-x_ref).abs().mean()), float((x-x_ref).abs().max()))


In [ ]:

plot_hist({"QP (k-hot)": x}, f"k-hot value distribution (log-y), sum={k}", bins=160, logy=True, xlim=(0.0, 1.0))
savefig("khot_hist_logy.png")
plt.show()

plot_log10_positive_with_zero_mass(x, f"k-hot: log10(value) with zero-mass separation (sum={k})", eps=cfg.eps_log)
savefig("khot_log10_positive.png")
plt.show()

plot_sorted_quantile_band(x, "k-hot: sorted profile quantile band (top-60)", k_max=60, logy=True)
savefig("khot_sorted_profiles_quantiles_top60.png")
plt.show()

plot_support_size(x, "k-hot: support size distribution")
savefig("khot_support_size.png")
plt.show()

err = (x - x_ref).abs()
plot_error_distributions(err, "k-hot: error |x_fast - x_ref| (fast vs high-iter)", eps=1e-18)
savefig("khot_error_improved.png")
plt.show()


## Use case 3 — Adaptive \(\xi\) (learnable mass)

\[
\xi = \sum_i m_i + \sigma(t)\left(\sum_i M_i - \sum_i m_i\right).
\]

With \(m=0\), \(M=1\), \(\xi\in[0,N]\).


In [ ]:

B, N = cfg.B, 80
z = torch.randn(B, N, dtype=cfg.dtype)

proj = SeparableQPProjection(
    n=N,
    bounds_mode="fixed",
    m_init=0.0,
    M_init=1.0,
    learn_xi=True,
    xi_mode="feasible_sigmoid",
    bisection_iters=60,
)

ts = torch.linspace(-6, 6, 121)
mean_sums, mean_L0 = [], []
for t in ts:
    with torch.no_grad():
        proj.t_raw[:] = torch.tensor([float(t.item())], dtype=torch.float32)
    x = proj(z)
    mean_sums.append(float(x.sum(dim=-1).mean().item()))
    mean_L0.append(float((x > 0).sum(dim=-1).float().mean().item()))

plt.figure(figsize=(8, 4.6))
plt.plot(to_np(ts), np.array(mean_sums), label="mean(sum x)")
plt.plot(to_np(ts), np.array(mean_L0), label="mean(support size)")
plt.title("Adaptive xi: effect of t on total mass and sparsity")
plt.xlabel("t")
plt.legend()
savefig("adaptive_xi_curve.png")
plt.show()


## Use case 4 — Learnable bounds (unit mode)

We learn per-coordinate \(m_i\) and \(M_i\) while guaranteeing \(0<m_i<M_i<1\) by construction.


In [ ]:

set_seed(1)
B, N = 128, 30
z = torch.randn(B, N, dtype=cfg.dtype)

target_layer = SeparableQPProjection(n=N, xi=1.0, bounds_mode="fixed", m_init=0.0, M_init=1.0, bisection_iters=140)
x_target = target_layer(z + 0.7).detach()

proj = SeparableQPProjection(
    n=N,
    xi=1.0,
    bounds_mode="unit",
    learn_bounds=True,
    clamp_xi_to_feasible=True,
    bisection_iters=60,
)

with torch.no_grad():
    m0, M0 = proj.bounds(dtype=cfg.dtype, device=torch.device(cfg.device))

opt = torch.optim.Adam(proj.parameters(), lr=3e-2)
losses = []
for _ in range(250):
    opt.zero_grad(set_to_none=True)
    x = proj(z)
    loss = F.mse_loss(x, x_target)
    loss.backward()
    opt.step()
    losses.append(float(loss.item()))

with torch.no_grad():
    m1, M1 = proj.bounds(dtype=cfg.dtype, device=torch.device(cfg.device))
    x_final = proj(z)
    err = (x_final - x_target).abs()

print("Final loss:", losses[-1])
print("Bounds init mean:", float(m0.mean()), float(M0.mean()), "width mean:", float((M0-m0).mean()))
print("Bounds final mean:", float(m1.mean()), float(M1.mean()), "width mean:", float((M1-m1).mean()))

plt.figure(figsize=(8,4.6))
plt.plot(losses)
plt.yscale("log")
plt.title("Learning bounds: loss curve (log-y)")
plt.xlabel("step")
savefig("learn_bounds_loss_logy.png")
plt.show()

plot_hist({"width_init": (M0-m0), "width_final": (M1-m1)}, "Width (M-m): init vs final (log-y)", bins=80, logy=True, xlim=(0.0, 1.0))
savefig("learn_bounds_width_compare_logy.png")
plt.show()

plot_error_distributions(err, "|x_final - x_target| distributions", eps=1e-18)
savefig("learn_bounds_abs_error_improved.png")
plt.show()


## Use case 5 — Gamma (curvature) effects

Compare uniform \(\gamma\) vs custom per-dimension \(\gamma\).


In [ ]:

set_seed(2)
B, N = cfg.B, 60
z = torch.randn(B, N, dtype=cfg.dtype)

proj_uniform = SeparableQPProjection(n=N, xi=1.0, bounds_mode="fixed", m_init=0.0, M_init=1.0, gamma_init=1.0)

gamma_vec = torch.ones(N)
gamma_vec[: N//3] = 0.2
gamma_vec[N//3 : 2*N//3] = 1.0
gamma_vec[2*N//3 :] = 5.0
proj_custom = SeparableQPProjection(
    n=N, xi=1.0, bounds_mode="fixed", m_init=0.0, M_init=1.0,
    gamma_init=gamma_vec, learn_gamma=False, gamma_per_dim=True
)

x_u = proj_uniform(z)
x_c = proj_custom(z)

def group_sums(x: torch.Tensor) -> torch.Tensor:
    return torch.stack([x[:, :N//3].sum(-1), x[:, N//3:2*N//3].sum(-1), x[:, 2*N//3:].sum(-1)], dim=-1)

gs_u = group_sums(x_u)
gs_c = group_sums(x_c)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for i, ax in enumerate(axes):
    ax.hist(to_np(gs_u[:, i]), bins=50, density=True, alpha=0.55, label="uniform")
    ax.hist(to_np(gs_c[:, i]), bins=50, density=True, alpha=0.55, label="custom")
    ax.set_title(f"group {i+1} mass")
    ax.set_xlim(0, 1.0)
    ax.legend()
fig.suptitle("Gamma effect: mass distribution by coordinate group")
fig.tight_layout()
path = Path(cfg.out_dir) / "gamma_group_mass_compare_improved.png"
fig.savefig(path, dpi=170)
print("saved:", path)
plt.show()

plot_support_size(x_u, "Support size (uniform gamma)")
savefig("gamma_support_size_uniform.png")
plt.show()

plot_support_size(x_c, "Support size (custom gamma)")
savefig("gamma_support_size_custom.png")
plt.show()


## Use case 6 — Solve from \(\beta\) (requested plot improvements)

Requested improvements applied:
- `solve_from_beta_hist_log10x_improved.png`
- `solve_from_beta_error_vs_high_iter_improved.png`


In [ ]:

set_seed(3)
B, N = 128, 40
beta = torch.randn(B, N, dtype=cfg.dtype)

proj_fast = SeparableQPProjection(n=N, xi=1.0, bounds_mode="fixed", m_init=0.0, M_init=1.0, bisection_iters=40)
proj_ref  = SeparableQPProjection(n=N, xi=1.0, bounds_mode="fixed", m_init=0.0, M_init=1.0, bisection_iters=260)

solver_fast = SeparableQPSolveFromBeta(projection=proj_fast)
solver_ref  = SeparableQPSolveFromBeta(projection=proj_ref)

x = solver_fast(beta)
x_ref = solver_ref(beta)

plot_log10_positive_with_zero_mass(x, "Solve-from-beta: log10(value) with zero-mass separation", eps=cfg.eps_log)
savefig("solve_from_beta_hist_log10x_improved.png")
plt.show()

err = (x - x_ref).abs()
plot_error_distributions(err, "Solve-from-beta: error |x_fast - x_ref| (fast vs high-iter)", eps=1e-18)
savefig("solve_from_beta_error_vs_high_iter_improved.png")
plt.show()


## Use case 7 — Fixed \(z\) (deterministic output)

If \(z\) is fixed, every batch row is identical; histograms show spikes.


In [ ]:

set_seed(4)
N = 50
z_const = torch.linspace(-1.5, 1.5, N)

proj = SeparableQPProjection(n=N, xi=1.0, bounds_mode="fixed", m_init=0.0, M_init=1.0, bisection_iters=120)
fixed = FixedZProjection(z_const=z_const, projection=proj)

x = fixed(batch_size=cfg.B, dtype=cfg.dtype)

plot_hist({"x": x}, "Fixed z: value distribution (log-y)", bins=80, logy=True, xlim=(0.0, 1.0))
savefig("fixed_z_hist_logy.png")
plt.show()

plot_sorted_quantile_band(x, "Fixed z: sorted profile quantile band (top-50)", k_max=50, logy=True)
savefig("fixed_z_sorted_profiles_quantiles.png")
plt.show()


## Geometry intuition (N=2)

Contours + feasible segment + \(z\) + \(x^\*\).


In [ ]:

set_seed(5)
z = torch.tensor([0.6, -0.2], dtype=torch.float32)
gamma = torch.tensor([0.7, 2.0], dtype=torch.float32)
m = torch.tensor([-0.3, -0.1], dtype=torch.float32)
M = torch.tensor([0.8, 0.6], dtype=torch.float32)
xi = 0.5

x1 = torch.linspace(float(m[0]), float(M[0]), 260)
x2 = torch.linspace(float(m[1]), float(M[1]), 260)
X1, X2 = torch.meshgrid(x1, x2, indexing="xy")
Fval = gamma[0] * (X1 - z[0]) ** 2 + gamma[1] * (X2 - z[1]) ** 2

layer = SeparableQPProjection(
    n=2, xi=xi, bounds_mode="fixed",
    m_init=m.numpy(), M_init=M.numpy(),
    gamma_init=gamma.numpy(), learn_gamma=False,
    bisection_iters=200
)
x_star = layer(z.reshape(1, 2)).detach().cpu().reshape(-1)

x1_seg = torch.linspace(float(m[0]), float(M[0]), 600)
x2_seg = xi - x1_seg
mask = (x2_seg >= m[1]) & (x2_seg <= M[1])

plt.figure(figsize=(7.2, 6.0))
plt.contour(to_np(X1), to_np(X2), to_np(Fval), levels=25)
plt.plot(to_np(x1_seg[mask]), to_np(x2_seg[mask]), linewidth=2.3, label="feasible (sum=xi)")
plt.scatter([z[0].item()], [z[1].item()], marker="x", s=80, label="unconstrained minimizer z")
plt.scatter([x_star[0].item()], [x_star[1].item()], s=80, label="projected solution x*")
plt.title("N=2 geometry: objective contours + feasible set + solution")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend()
savefig("contour_n2_improved.png")
plt.show()
